In [15]:
# %%
# Step 1: Import libraries
# -----------------------
# Import required libraries for data manipulation and financial calculations.

import pandas as pd
import numpy as np
from numpy_financial import npv, irr
import ast

In [16]:
# %%
# Step 2: Load main EPC dataset
# -----------------------------
# Load the processed EPC data containing improvement metrics.

data_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_step1_improvement_metrics.csv"
df = pd.read_csv(data_path)
df.shape

(117, 130)

In [17]:
# %%
# Step 3: Inspect raw improvements data
# -------------------------------------
# Look at the first few rows of the "IMPROVEMENTS_PARSED" column.

df["IMPROVEMENTS_PARSED"].head(3)

0    [{'description': 'Solar water heating', 'cost_...
1    [{'description': 'Floor insulation (suspended ...
2    [{'description': 'Solar water heating', 'cost_...
Name: IMPROVEMENTS_PARSED, dtype: object

In [18]:
# %%
# Step 4: Parse improvements into lists
# -------------------------------------
# Convert string representations of lists into actual Python lists.

def parse_improvements(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except:
        return []

df["IMPROVEMENTS_LIST"] = df["IMPROVEMENTS_PARSED"].apply(parse_improvements)

In [19]:
# %%
# Step 5: Aggregate costs per dwelling
# ------------------------------------
# Sum minimum, maximum, and mid CAPEX across all improvements in a dwelling.

def aggregate_costs(improvements):
    cost_min = 0
    cost_max = 0
    for imp in improvements:
        if isinstance(imp, dict):
            if imp.get("cost_min") is not None:
                cost_min += imp["cost_min"]
            if imp.get("cost_max") is not None:
                cost_max += imp["cost_max"]
    return pd.Series({
        "CAPEX_MIN": cost_min if cost_min > 0 else np.nan,
        "CAPEX_MAX": cost_max if cost_max > 0 else np.nan,
        "CAPEX_MID": np.mean([cost_min, cost_max]) if cost_min > 0 and cost_max > 0 else np.nan
    })

df[["CAPEX_MIN", "CAPEX_MAX", "CAPEX_MID"]] = df["IMPROVEMENTS_LIST"].apply(aggregate_costs)

In [20]:
# %%
# Step 6: Load measure lifetimes
# ------------------------------
# Load a CSV mapping individual measures to realistic lifetimes.

lifetimes_path = "/Users/jakegehrung/Desktop/data_raw/EEMs/measures_lifetime.csv"
df_lifetimes = pd.read_csv(lifetimes_path)
df_lifetimes["DATASET_DESCRIPTION"] = df_lifetimes["DATASET_DESCRIPTION"].str.lower()

In [21]:
# %%
# %%
# Step 7 (robust version): Map improvements to lifetimes safely
# -------------------------------------------------------------
# Computes dynamic project lifetime per dwelling using weighted or minimum cost weighting.

# Define a default project lifetime in years for dwellings with missing or invalid improvements
PROJECT_LIFETIME_DEFAULT = 15  # You can adjust this value as needed

def get_lifetimes_robust(improvements, method="weighted"):
    """
    Calculate a dynamic project lifetime based on individual measure lifetimes.
    
    Parameters:
        improvements (list of dict): List of improvement dictionaries.
        method (str): 'weighted' (default) or 'minimum' lifetime calculation.
        
    Returns:
        float: Computed project lifetime in years.
    """
    lifetimes = []
    costs = []
    
    if not isinstance(improvements, list):
        return PROJECT_LIFETIME_DEFAULT  # fallback if not a list
    
    for imp in improvements:
        if isinstance(imp, dict) and imp is not None:
            desc = str(imp.get("description", "")).lower().strip()
            if not desc:  # skip empty descriptions
                continue
            
            # compute midpoint cost
            cost_min = imp.get("cost_min") or 0
            cost_max = imp.get("cost_max") or 0
            cost_mid = np.mean([cost_min, cost_max])
            
            # lookup lifetime in lifetimes dataframe
            match = df_lifetimes.loc[df_lifetimes["DATASET_DESCRIPTION"] == desc, "LIFETIME"]
            if not match.empty:
                lifetimes.append(match.values[0])
                costs.append(cost_mid)
    
    if not lifetimes:  # fallback if no valid lifetimes found
        return PROJECT_LIFETIME_DEFAULT
    
    lifetimes_array = np.array(lifetimes)
    
    if method == "weighted":
        costs_array = np.array(costs)
        if costs_array.sum() > 0:
            return np.average(lifetimes_array, weights=costs_array)
        else:
            return lifetimes_array.mean()  # fallback to unweighted average
    elif method == "minimum":
        return lifetimes_array.min()
    else:
        raise ValueError("method must be 'weighted' or 'minimum'")

# Apply the robust function to calculate dynamic project lifetimes
df["PROJECT_LIFETIME_DYNAMIC"] = df["IMPROVEMENTS_LIST"].apply(
    lambda x: get_lifetimes_robust(x, method="weighted")
)

In [22]:
# %%
# Step 8: Assign annual cashflow
# ------------------------------
# Use reported annual energy cost savings as the annual cashflow for NPV/IRR.

df["ANNUAL_CASHFLOW"] = df["ANNUAL_ENERGY_COST_SAVINGS"]

In [23]:
# %%
# %%
# Step 9: Calculate NPV using dynamic lifetimes
# ---------------------------------------------
# Compute Net Present Value per dwelling using the dynamic project lifetime.

# Define discount rate (annual) for NPV calculations
DISCOUNT_RATE = 0.05  # 5% annual discount rate; adjust as needed

def calculate_npv_dynamic(capex, annual_cashflow, lifetime):
    """
    Compute NPV for a dwelling given CAPEX, annual cashflow, and project lifetime.
    
    Parameters:
        capex (float): Total investment cost.
        annual_cashflow (float): Annual savings from improvements.
        lifetime (float): Project lifetime in years.
        
    Returns:
        float: Net Present Value (NPV)
    """
    if pd.isna(capex) or pd.isna(annual_cashflow) or annual_cashflow <= 0:
        return np.nan
    
    # Create cashflow list: initial investment negative, then annual savings
    cashflows = [-capex] + [annual_cashflow] * int(lifetime)
    
    # Compute NPV
    return npv(DISCOUNT_RATE, cashflows)

# Apply function to dataframe
df["NPV"] = df.apply(
    lambda r: calculate_npv_dynamic(r["CAPEX_MID"], r["ANNUAL_CASHFLOW"], r["PROJECT_LIFETIME_DYNAMIC"]),
    axis=1
)

In [24]:
# %%
# Step 10: Calculate IRR using dynamic lifetimes
# ----------------------------------------------
# Compute Internal Rate of Return per dwelling.

def calculate_irr_dynamic(capex, annual_cashflow, lifetime):
    if pd.isna(capex) or pd.isna(annual_cashflow) or annual_cashflow <= 0:
        return np.nan
    cashflows = [-capex] + [annual_cashflow] * int(lifetime)
    try:
        return irr(cashflows)
    except:
        return np.nan

df["IRR"] = df.apply(
    lambda r: calculate_irr_dynamic(r["CAPEX_MID"], r["ANNUAL_CASHFLOW"], r["PROJECT_LIFETIME_DYNAMIC"]),
    axis=1
)

In [25]:
# %%
# Step 11: Simple payback and cost-effectiveness metrics
# ------------------------------------------------------
# Calculate simple payback, cost per kWh reduced, and cost per ton CO2 reduced.

df["SIMPLE_PAYBACK_YEARS"] = df["CAPEX_MID"] / df["ANNUAL_CASHFLOW"]

df["COST_PER_KWH_REDUCED"] = df["CAPEX_MID"] / df["DELTA_ENERGY_CONSUMPTION"]

df["COST_PER_TON_CO2_REDUCED"] = df["CAPEX_MID"] / (df["DELTA_CO2_EMISSIONS"] / 1000)

In [26]:
# %%
# Step 12: Flag dwellings with valid financial data
# -------------------------------------------------
# Identify which dwellings have enough data to compute financial metrics.

df["FINANCIAL_DATA_AVAILABLE"] = (
    df["CAPEX_MID"].notna() &
    df["ANNUAL_CASHFLOW"].notna()
)

df["ECONOMICALLY_POSITIVE"] = (
    (df["NPV"] > 0) |
    (df["IRR"] > DISCOUNT_RATE)
)

In [27]:
# %%
# Step 13: Generate financial summary
# -----------------------------------
# Produce descriptive statistics for all financial metrics.

financial_summary = [
    "CAPEX_MID",
    "ANNUAL_CASHFLOW",
    "NPV",
    "IRR",
    "SIMPLE_PAYBACK_YEARS",
    "COST_PER_KWH_REDUCED",
    "COST_PER_TON_CO2_REDUCED"
]

df[financial_summary].describe().T

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,count,mean,std,min,25%,50%,75%,max
CAPEX_MID,112.0,1.827688e+04,1.122386e+04,2.250000e+01,1.004125e+04,1.450000e+04,2.686250e+04,4.722500e+04
ANNUAL_CASHFLOW,117.0,5.946923e+02,7.036761e+02,0.000000e+00,1.270000e+02,3.240000e+02,8.060000e+02,3.755000e+03
NPV,110.0,-8.310723e+03,1.336772e+04,-3.387692e+04,-1.693425e+04,-7.570365e+03,-6.960487e+02,4.133593e+04
IRR,110.0,1.370675e-02,1.667555e-01,-1.220550e-01,-5.245576e-02,-7.063066e-03,3.929720e-02,1.599999e+00
SIMPLE_PAYBACK_YEARS,112.0,inf,NaN,6.250000e-01,1.892865e+01,3.912069e+01,8.574330e+01,inf
COST_PER_KWH_REDUCED,112.0,inf,NaN,1.406250e+00,8.718203e+01,1.310772e+02,1.993198e+02,inf
COST_PER_TON_CO2_REDUCED,112.0,5.607786e+06,2.573258e+07,-1.475000e+08,2.473361e+06,4.150974e+06,9.455769e+06,7.375000e+07


In [28]:
# %%
# Step 14: Save updated dataset
# -----------------------------
# Save the EPC dataset with all financial metrics and dynamic lifetimes applied.

#output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_step2_financial_metrics.csv"
#df.to_csv(output_path, index=False)
#output_path